<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/Conteo_de_Veh%C3%ADculos_con_RF_DETR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# INSTALAR RF-DETR Y SUPERVISION
# ============================================================

!pip install -q rfdetr supervision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 602.7/602.7 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 391.6/391.6 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 11.6 MB/s eta 0:00:00


In [ ]:
import cv2
import numpy as np
import supervision as sv

from rfdetr import RFDETRMedium
from rfdetr.assets.coco_classes import COCO_CLASSES
from collections import defaultdict
from google.colab.patches import cv2_imshow
from google.colab import files

In [ ]:
# ============================================================
# CONFIGURACIÓN
# ============================================================

VIDEO_PATH = "video1.mp4"

# RF-DETR preentrenado en COCO
# Opciones disponibles: RFDETRNano, RFDETRSmall,
# RFDETRMedium y RFDETRLarge
MODEL_SIZE = "medium"

OUTPUT_PATH = "resultado_vehiculos_rfdetr_bytetrack.mp4"

CONFIDENCE = 0.40

In [ ]:
CLASSES_TO_COUNT = {
    2: "Persona",
    3: "Auto",
    4: "Motocicleta",
    6: "Autobus",
    8: "Camion"
}

In [ ]:
# ============================================================
# REGIÓN DE INTERÉS
# ============================================================

REGION = np.array([
    [500, 150],
    [1800, 150],
    [1800, 650],
    [500, 650]
], dtype=np.int32)

In [ ]:
# ============================================================
# PARÁMETROS DE ESTABILIDAD
# ============================================================

# Frames consecutivos que un objeto debe estar
# dentro de la región antes de considerarlo válido
MIN_FRAMES_TO_CONFIRM = 10


# Número de frames que ByteTrack conserva
# un objeto cuando temporalmente no lo detecta
TRACK_BUFFER = 60

In [ ]:
# ============================================================
# MODELO RF-DETR
# ============================================================

print("Cargando RF-DETR...")

if MODEL_SIZE == "medium":
    model = RFDETRMedium()
elif MODEL_SIZE == "nano":
    from rfdetr import RFDETRNano
    model = RFDETRNano()
elif MODEL_SIZE == "small":
    from rfdetr import RFDETRSmall
    model = RFDETRSmall()
elif MODEL_SIZE == "large":
    from rfdetr import RFDETRLarge
    model = RFDETRLarge()
else:
    raise ValueError("MODEL_SIZE debe ser: nano, small, medium o large")

print("Modelo RF-DETR cargado.")

Cargando RF-DETR...
[2026-09-23 17:27:43] [INFO] rf-detr - Downloading pretrained weights for /root/.roboflow/models/rf-detr-medium.pth


/root/.roboflow/models/rf-detr-medium.pth:   0%|          | 0.00/386M [00:00<?, ?iB/s]

[2026-09-23 17:27:55] [INFO] rf-detr - MD5 validation successful for /root/.roboflow/models/rf-detr-medium.pth


[2026-09-23 17:27:55] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-23 17:27:55] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-09-23 17:27:57] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-medium.pth already exists with correct MD5 hash.
Modelo RF-DETR cargado.


In [ ]:
# ============================================================
# BYTE TRACK
# ============================================================

tracker = sv.ByteTrack(
    track_activation_threshold=0.25,
    lost_track_buffer=TRACK_BUFFER,
    minimum_matching_threshold=0.8,
    frame_rate=30
)

/tmp/ipykernel_1260/2498558552.py:5: FutureWarning: The `ByteTrack` was deprecated since v0.28.0. It will be removed in v0.31.0.
  tracker = sv.ByteTrack(


In [ ]:
# ============================================================
# VIDEO
# ============================================================

cap = cv2.VideoCapture(
    VIDEO_PATH
)

if not cap.isOpened():

    raise Exception(
        "No se pudo abrir el video"
    )

In [ ]:
# ============================================================
# INFORMACIÓN DEL VIDEO
# ============================================================

fps = cap.get(
    cv2.CAP_PROP_FPS
)

width = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

height = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)

total_frames = int(
    cap.get(
        cv2.CAP_PROP_FRAME_COUNT
    )
)


print()
print("Información del video")
print("---------------------")
print(f"Resolución: {width}x{height}")
print(f"FPS: {fps:.2f}")
print(f"Frames: {total_frames}")


Información del video
---------------------
Resolución: 1920x1080
FPS: 30.00
Frames: 899


In [ ]:
!ffmpeg -i "video1.mp4" -r 25 -c:v libx264 -pix_fmt yuv420p "video1_25fps.mp4"

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [ ]:
# ============================================================
# VIDEO DE SALIDA
# ============================================================

fourcc = cv2.VideoWriter_fourcc(
    *"mp4v"
)

out = cv2.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (width, height)
)

In [ ]:
# ============================================================
# VARIABLES DEL CONTADOR
# ============================================================

# IDs que ya fueron contados
counted_ids = set()


# Cuántos frames lleva cada ID dentro
# de la región
inside_frames = defaultdict(int)


# Última posición conocida
last_positions = {}


# Clase asociada al ID
track_classes = {}


# Conteo por clase
total_count = defaultdict(int)


# Objetos actualmente dentro
inside_ids = set()

In [ ]:
# ============================================================
# FUNCIÓN ROI
# ============================================================

def point_inside_region(
    point,
    polygon
):

    x, y = point

    return cv2.pointPolygonTest(
        polygon,
        (
            float(x),
            float(y)
        ),
        False
    ) >= 0


In [ ]:
# ============================================================
# DASHBOARD
# ============================================================

def draw_dashboard(
    frame,
    total_count,
    inside_ids,
    counted_ids
):

    h, w = frame.shape[:2]

    dashboard_height = 125

    overlay = frame.copy()

    cv2.rectangle(
        overlay,
        (0, 0),
        (w, dashboard_height),
        (15, 15, 15),
        -1
    )

    frame = cv2.addWeighted(
        overlay,
        0.85,
        frame,
        0.15,
        0
    )


    # --------------------------------------------------------
    # TÍTULO
    # --------------------------------------------------------

    cv2.putText(
        frame,
        "VEHICLE TRACKING",
        (20, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )


    # --------------------------------------------------------
    # TOTAL
    # --------------------------------------------------------

    total = sum(
        total_count.values()
    )

    cv2.putText(
        frame,
        f"TOTAL: {total}",
        (20, 70),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 255),
        2
    )


    # --------------------------------------------------------
    # DENTRO
    # --------------------------------------------------------

    cv2.putText(
        frame,
        f"DENTRO: {len(inside_ids)}",
        (220, 70),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )


    # --------------------------------------------------------
    # IDS
    # --------------------------------------------------------

    cv2.putText(
        frame,
        f"IDS: {len(counted_ids)}",
        (440, 70),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 180, 0),
        2
    )


    # --------------------------------------------------------
    # CLASES
    # --------------------------------------------------------

    x = 20

    y = 110

    for clase, cantidad in total_count.items():

        texto = (
            f"{clase}: {cantidad}"
        )

        cv2.putText(
            frame,
            texto,
            (x, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            1
        )

        x += 160


    return frame

In [ ]:
# ============================================================
# PROCESAMIENTO CON RF-DETR + BYTE TRACK
# ============================================================

frame_number = 0

print()
print("Procesando video con RF-DETR...")
print()

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    # ========================================================
    # RF-DETR
    # ========================================================
    # OpenCV entrega BGR.
    # RF-DETR espera una imagen RGB.

    frame_rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    detections = model.predict(
        frame_rgb,
        threshold=CONFIDENCE
    )

    # RF-DETR ya devuelve sv.Detections,
    # por lo que NO necesitamos:
    # sv.Detections.from_ultralytics()

    # ========================================================
    # FILTRAR VEHÍCULOS
    # ========================================================

    if len(detections) > 0:

        mask = np.array([
            int(class_id) in CLASSES_TO_COUNT
            for class_id in detections.class_id
        ])

        detections = detections[mask]

    # ========================================================
    # BYTE TRACK
    # ========================================================

    detections = tracker.update_with_detections(
        detections
    )

    current_inside_ids = set()

    # ========================================================
    # PROCESAR TRACKS
    # ========================================================

    for i in range(len(detections)):

        xyxy = detections.xyxy[i]

        class_id = int(
            detections.class_id[i]
        )

        tracker_id = int(
            detections.tracker_id[i]
        )

        x1, y1, x2, y2 = map(
            int,
            xyxy
        )

        # ====================================================
        # CENTRO INFERIOR
        # ====================================================

        center_x = int(
            (x1 + x2) / 2
        )

        center_y = int(
            y2
        )

        center = (
            center_x,
            center_y
        )

        # Guardar posición
        last_positions[tracker_id] = center

        # Guardar clase
        track_classes[tracker_id] = class_id

        # ====================================================
        # ¿ESTÁ DENTRO DEL ROI?
        # ====================================================

        inside = point_inside_region(
            center,
            REGION
        )

        if inside:

            current_inside_ids.add(
                tracker_id
            )

            # -----------------------------------------------
            # CONFIRMACIÓN DEL TRACK
            # -----------------------------------------------

            inside_frames[tracker_id] += 1

            # -----------------------------------------------
            # CONTAR SOLO CUANDO ESTÁ CONFIRMADO
            # -----------------------------------------------

            if inside_frames[tracker_id] >= MIN_FRAMES_TO_CONFIRM:

                if tracker_id not in counted_ids:

                    counted_ids.add(
                        tracker_id
                    )

                    class_name = CLASSES_TO_COUNT[class_id]

                    total_count[class_name] += 1

        # Si sale del ROI no reiniciamos
        # el ID ya contabilizado.

        # ====================================================
        # COLOR
        # ====================================================

        if tracker_id in counted_ids:

            color = (
                0,
                255,
                0
            )

        elif inside:

            color = (
                0,
                255,
                255
            )

        else:

            color = (
                0,
                165,
                255
            )

        # ====================================================
        # BOUNDING BOX
        # ====================================================

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            color,
            2
        )

        # ====================================================
        # LABEL
        # ====================================================

        class_name = CLASSES_TO_COUNT[class_id]

        label = (
            f"{class_name} "
            f"ID:{tracker_id}"
        )

        cv2.putText(
            frame,
            label,
            (
                x1,
                max(
                    y1 - 10,
                    20
                )
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            color,
            2
        )

        # ====================================================
        # CENTRO
        # ====================================================

        cv2.circle(
            frame,
            center,
            5,
            color,
            -1
        )

    # ========================================================
    # ACTUALIZAR OBJETOS DENTRO
    # ========================================================

    inside_ids = current_inside_ids

    # ========================================================
    # DIBUJAR ROI
    # ========================================================

    cv2.polylines(
        frame,
        [REGION],
        True,
        (255, 0, 255),
        3
    )

    # ========================================================
    # TEXTO ROI
    # ========================================================

    x_roi, y_roi = REGION[0]

    cv2.putText(
        frame,
        "REGION DE CONTEO",
        (
            x_roi,
            max(
                y_roi - 10,
                20
            )
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 0, 255),
        2
    )

    # ========================================================
    # DASHBOARD
    # ========================================================

    frame = draw_dashboard(
        frame,
        total_count,
        inside_ids,
        counted_ids
    )

    # ========================================================
    # GUARDAR FRAME
    # ========================================================

    out.write(
        frame
    )

    # ========================================================
    # PROGRESO
    # ========================================================

    if frame_number % 100 == 0:

        progress = (
            frame_number /
            total_frames
        ) * 100

        print(
            f"Procesado: "
            f"{frame_number}/"
            f"{total_frames} "
            f"({progress:.1f}%)"
        )

# ============================================================
# CERRAR
# ============================================================

cap.release()
out.release()

print()
print("Procesamiento terminado.")


Procesando video con RF-DETR...

Procesado: 100/899 (11.1%)
Procesado: 200/899 (22.2%)
Procesado: 300/899 (33.4%)
Procesado: 400/899 (44.5%)
Procesado: 500/899 (55.6%)
Procesado: 600/899 (66.7%)
Procesado: 700/899 (77.9%)
Procesado: 800/899 (89.0%)

Procesamiento terminado.


In [ ]:
# ============================================================
# RESULTADO FINAL
# ============================================================

print()
print("=" * 55)
print("        RESULTADO FINAL - RF-DETR + BYTE TRACK")
print("=" * 55)

print()

print(
    f"Total vehículos: "
    f"{sum(total_count.values())}"
)

print()

for clase, cantidad in total_count.items():

    print(
        f"{clase}: {cantidad}"
    )

print()

print(
    f"IDs contabilizados: "
    f"{len(counted_ids)}"
)

print()

print(
    f"Video generado: "
    f"{OUTPUT_PATH}"
)

print("=" * 55)

In [ ]:
# ============================================================
# DESCARGAR
# ============================================================

files.download(
    OUTPUT_PATH
)

POR SI EXISTE ERROR CON EL VIDEO EN EL USO DE GPU

In [ ]:
!pip install --force-reinstall "nvidia-cuda-nvrtc==13.0.88"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 10.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-cuda-nvrtc
    Found existing installation: nvidia-cuda-nvrtc 13.4.92
    Uninstalling nvidia-cuda-nvrtc-13.4.92:
      Successfully uninstalled nvidia-cuda-nvrtc-13.4.92


In [ ]:
import importlib.metadata as md

print("NVRTC:", md.version("nvidia-cuda-nvrtc"))

NVRTC: 13.0.88


In [ ]:
!find /usr/local/lib/python3.13/dist-packages/nvidia -name "libnvrtc*" 2>/dev/null

/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libnvrtc.alt.so.13
/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libnvrtc-builtins.so.13.0
/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libnvrtc-builtins.alt.so.13.0
/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libnvrtc.so.13
/usr/local/lib/python3.13/dist-packages/nvidia/cuda_nvrtc/lib/libnvrtc.so.12
/usr/local/lib/python3.13/dist-packages/nvidia/cuda_nvrtc/lib/libnvrtc-builtins.alt.so.12.9
/usr/local/lib/python3.13/dist-packages/nvidia/cuda_nvrtc/lib/libnvrtc.alt.so.12
/usr/local/lib/python3.13/dist-packages/nvidia/cuda_nvrtc/lib/libnvrtc-builtins.so.12.9


In [ ]:
import glob

archivos = glob.glob(
    "/usr/local/lib/python3.13/dist-packages/nvidia/**/libnvrtc-builtins.so*",
    recursive=True
)

for f in archivos:
    print(f)

/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libnvrtc-builtins.so.13.0
/usr/local/lib/python3.13/dist-packages/nvidia/cuda_nvrtc/lib/libnvrtc-builtins.so.12.9


In [ ]:
import os

cuda13_lib = "/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib"

os.environ["LD_LIBRARY_PATH"] = (
    cuda13_lib + ":" +
    os.environ.get("LD_LIBRARY_PATH", "")
)

print("CUDA 13 library:")
print(cuda13_lib)

CUDA 13 library:
/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib


In [ ]:
!ls -lh /usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/

total 1.8G
drwxr-xr-x 6 root root  4.0K Sep 22 13:31 cmake
-rw-r--r-- 1 root root  1.4M Sep 22 13:26 libcheckpoint.so
-rw-r--r-- 1 root root  517M Sep 22 13:26 libcublasLt.so.13
-rw-r--r-- 1 root root   52M Sep 22 13:26 libcublas.so.13
-rw-r--r-- 1 root root 1001K Sep 22 13:26 libcudadevrt.a
-rw-r--r-- 1 root root  688K Sep 22 13:26 libcudart.so.13
-rw-r--r-- 1 root root  1.3M Sep 22 13:26 libcudart_static.a
-rw-r--r-- 1 root root  274M Sep 22 13:26 libcufft.so.12
-rw-r--r-- 1 root root  969K Sep 22 13:26 libcufftw.so.12
-rw-r--r-- 1 root root   43K Sep 22 13:26 libcufile_rdma.so.1
-rw-r--r-- 1 root root  3.1M Sep 22 13:26 libcufile.so.0
-rw-r--r-- 1 root root  4.0M Sep 22 13:26 libcupti.so.13
-rw-r--r-- 1 root root  127M Sep 22 13:26 libcurand.so.10
-rw-r--r-- 1 root root  100M Sep 22 13:26 libcusolverMg.so.12
-rw-r--r-- 1 root root  135M Sep 22 13:26 libcusolver.so.12
-rw-r--r-- 1 root root  156M Sep 22 13:26 libcusparse.so.12
-rw-r--r-- 1 root root  721K Sep 22 13:26 libnvblas.so.13

In [ ]:
import ctypes

lib = "/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libnvrtc-builtins.so.13.0"

ctypes.CDLL(lib)

print("✅ NVRTC builtins 13.0 se puede cargar")

✅ NVRTC builtins 13.0 se puede cargar


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

spatial_shapes = torch.tensor(
    [[135, 240],
     [68, 120],
     [34, 60],
     [17, 30]],
    device="cuda",
    dtype=torch.int64
)

print(spatial_shapes)
print(spatial_shapes.prod(1))

PyTorch: 2.11.0+cu130
CUDA: 13.0
GPU: Tesla T4
tensor([[135, 240],
        [ 68, 120],
        [ 34,  60],
        [ 17,  30]], device='cuda:0')
tensor([32400,  8160,  2040,   510], device='cuda:0')


In [ ]:
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

detections = model.predict(
    frame_rgb,
    threshold=CONFIDENCE
)

print(detections)

Detections(xyxy=array([[1.62559296e+02, 5.62300598e+02, 5.24997498e+02, 8.37022644e+02],
       [2.59877869e+02, 7.30134155e+02, 6.90142151e+02, 1.03674548e+03],
       [1.14042969e+03, 1.68644745e+02, 1.34699829e+03, 3.41439087e+02],
       [0.00000000e+00, 8.45545776e+02, 3.40807190e+02, 1.07892700e+03],
       [7.72085266e+02, 1.55197617e+02, 9.71799988e+02, 3.17939545e+02],
       [2.23044052e+02, 4.42390594e+02, 5.40528687e+02, 6.23385864e+02],
       [8.43576843e+02, 4.42889771e+02, 1.15263525e+03, 7.01611206e+02],
       [2.99854675e+02, 3.88380859e+02, 5.64640320e+02, 6.06716064e+02],
       [7.86204590e+02, 3.21065369e+02, 1.05801428e+03, 5.42801025e+02],
       [2.37206970e+02, 3.12942871e+02, 4.99504852e+02, 5.14519897e+02],
       [6.75405731e+01, 2.80296051e+02, 3.01539185e+02, 5.05753967e+02],
       [1.47322906e+02, 9.00281616e+02, 2.44123276e+02, 9.75536133e+02],
       [1.07500916e+03, 2.77007163e-01, 1.23004883e+03, 5.55662422e+01],
       [3.18676025e+02, 1.17127365e